# Intent Resolution Evaluator

### Getting Started

This sample demonstrates how to use Intent Resolution Evaluator
Before running the sample:
```bash
pip install azure-ai-projects azure-identity azure-ai-evaluation
```
Set these environment variables with your own values:
1) **PROJECT_CONNECTION_STRING** - The project connection string, as found in the overview page of your Azure AI Foundry project.
2) **MODEL_DEPLOYMENT_NAME** - The deployment name of the AI model, as found under the "Name" column in the "Models + endpoints" tab in your Azure AI Foundry project.
3) **AZURE_OPENAI_ENDPOINT** - Azure Open AI Endpoint to be used for evaluation.
4) **AZURE_OPENAI_API_KEY** - Azure Open AI Key to be used for evaluation.
5) **AZURE_OPENAI_API_VERSION** - Azure Open AI Api version to be used for evaluation.
6) **AZURE_SUBSCRIPTION_ID** - Azure Subscription Id of Azure AI Project
7) **PROJECT_NAME** - Azure AI Project Name
8) **RESOURCE_GROUP_NAME** - Azure AI Project Resource Group Name


The Intent Resolution evaluator measures how well an agent has identified and resolved the user intent.
The scoring is on a 1-5 integer scale and is as follows:

  - Score 1: Response completely unrelated to user intent
  - Score 2: Response minimally relates to user intent
  - Score 3: Response partially addresses the user intent but lacks complete details
  - Score 4: Response addresses the user intent with moderate accuracy but has minor inaccuracies or omissions
  - Score 5: Response directly addresses the user intent and fully resolves it

The evaluation requires the following inputs:

  - Query    : The user query. Either a string with a user request or a list of messages with previous requests from the user and responses from the assistant, potentially including a system message.
  - Response : The response to be evaluated. Either a string or a message with the response from the agent to the last user query.

There is a third optional parameter:
  - ToolDefinitions : The list of tool definitions the agent can call. This may be useful for the evaluator to better assess if the right tool was called to resolve a given intent.

### Initialize Intent Resolution Evaluator


In [1]:
import os
from azure.ai.evaluation import AzureOpenAIModelConfiguration
from azure.identity import DefaultAzureCredential
from azure.ai.evaluation import IntentResolutionEvaluator
from pprint import pprint

model_config = AzureOpenAIModelConfiguration(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
    azure_deployment=os.environ["MODEL_DEPLOYMENT_NAME"],
)

intent_resolution_evaluator = IntentResolutionEvaluator(model_config)

Class IntentResolutionEvaluator: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


### Samples

#### Evaluating query and response as string

In [2]:
#Success example. Intent is identified and understood and the response correctly resolves user intent
result = intent_resolution_evaluator(query="What are the opening hours of the Eiffel Tower?",
                                     response="Opening hours of the Eiffel Tower are 9:00 AM to 11:00 PM.",
                                    )
pprint(result)

Conversation history could not be parsed, falling back to original query: What are the opening hours of the Eiffel Tower?
Empty agent response extracted, likely due to input schema change. Falling back to using the original response: Opening hours of the Eiffel Tower are 9:00 AM to 11:00 PM.
Empty agent response extracted, likely due to input schema change. Falling back to using the original response: Opening hours of the Eiffel Tower are 9:00 AM to 11:00 PM.


{'gpt_intent_resolution': 5.0,
 'intent_resolution': 5.0,
 'intent_resolution_completion_tokens': 45,
 'intent_resolution_finish_reason': 'stop',
 'intent_resolution_model': 'gpt-4.1-nano-2025-04-14',
 'intent_resolution_prompt_tokens': 1907,
 'intent_resolution_reason': "The user asked for the Eiffel Tower's opening "
                             'hours. The agent provided a clear, accurate '
                             'answer that fully resolves the intent without '
                             'omissions or errors.',
 'intent_resolution_result': 'pass',
 'intent_resolution_sample_input': '[{"role": "user", "content": '
                                   '"{\\"query\\": \\"What are the opening '
                                   'hours of the Eiffel Tower?\\", '
                                   '\\"response\\": \\"Opening hours of the '
                                   'Eiffel Tower are 9:00 AM to 11:00 PM.\\", '
                                   '\\"tool_definitions\\": null

In [3]:
#Failure example. Even though intent is correctly identified, the response does not resolve the user intent
result = intent_resolution_evaluator(query="What is the opening hours of the Eiffel Tower?",
                                     response="Please check the official website for the up-to-date information on Eiffel Tower opening hours.",
                                    )
pprint(result)

Conversation history could not be parsed, falling back to original query: What is the opening hours of the Eiffel Tower?
Empty agent response extracted, likely due to input schema change. Falling back to using the original response: Please check the official website for the up-to-date information on Eiffel Tower opening hours.
Empty agent response extracted, likely due to input schema change. Falling back to using the original response: Please check the official website for the up-to-date information on Eiffel Tower opening hours.


{'gpt_intent_resolution': 3.0,
 'intent_resolution': 3.0,
 'intent_resolution_completion_tokens': 61,
 'intent_resolution_finish_reason': 'stop',
 'intent_resolution_model': 'gpt-4.1-nano-2025-04-14',
 'intent_resolution_prompt_tokens': 1905,
 'intent_resolution_reason': "The user asked for Eiffel Tower's opening hours. "
                             'The agent directed the user to check the '
                             'official website, which is a relevant but '
                             'indirect response; it does not provide the '
                             'requested information directly, resulting in a '
                             'mostly incomplete resolution.',
 'intent_resolution_result': 'pass',
 'intent_resolution_sample_input': '[{"role": "user", "content": '
                                   '"{\\"query\\": \\"What is the opening '
                                   'hours of the Eiffel Tower?\\", '
                                   '\\"response\\": \\"Please ch

#### Evaluating query and response as list of messages

In [4]:
query = [
    {
        "role": "system",
        "content": "You are a friendly and helpful customer service agent."
    },
    {
        "createdAt": "2025-03-14T06:14:20Z",
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "Hi, I need help with the last 2 orders on my account #888. Could you please update me on their status?"
            }
        ]
    }
]

response = [
    {
        "createdAt": "2025-03-14T06:14:30Z",
        "run_id": "0",
        "role": "assistant",
        "content": [
            {
                "type": "text",
                "text": "Hello! Let me quickly look up your account details."
            }
        ]
    },
    {
        "createdAt": "2025-03-14T06:14:35Z",
        "run_id": "0",
        "role": "assistant",
        "content": [
            {
                "type": "tool_call",
                "tool_call_id": "tool_call_20250310_001",
                "name": "get_orders",
                "arguments": {
                    "account_number": "888"
                }
            }
        ]
    },
    {
        "createdAt": "2025-03-14T06:14:40Z",
        "run_id": "0",
        "tool_call_id": "tool_call_20250310_001",
        "role": "tool",
        "content": [
            {
                "type": "tool_result",
                "tool_result": "[{ \"order_id\": \"123\" }, { \"order_id\": \"124\" }]"
            }
        ]
    },
    {
        "createdAt": "2025-03-14T06:14:45Z",
        "run_id": "0",
        "role": "assistant",
        "content": [
            {
                "type": "text",
                "text": "Thanks for your patience. I see two orders on your account. Let me fetch the details for both."
            }
        ]
    },
    {
        "createdAt": "2025-03-14T06:14:50Z",
        "run_id": "0",
        "role": "assistant",
        "content": [
            {
                "type": "tool_call",
                "tool_call_id": "tool_call_20250310_002",
                "name": "get_order",
                "arguments": {
                    "order_id": "123"
                }
            },
            {
                "type": "tool_call",
                "tool_call_id": "tool_call_20250310_003",
                "name": "get_order",
                "arguments": {
                    "order_id": "124"
                }
            }
        ]
    },
    {
        "createdAt": "2025-03-14T06:14:55Z",
        "run_id": "0",
        "tool_call_id": "tool_call_20250310_002",
        "role": "tool",
        "content": [
            {
                "type": "tool_result",
                "tool_result": "{ \"order\": { \"id\": \"123\", \"status\": \"shipped\", \"delivery_date\": \"2025-03-15\" } }"
            }
        ]
    },
    {
        "createdAt": "2025-03-14T06:15:00Z",
        "run_id": "0",
        "tool_call_id": "tool_call_20250310_003",
        "role": "tool",
        "content": [
            {
                "type": "tool_result",
                "tool_result": "{ \"order\": { \"id\": \"124\", \"status\": \"delayed\", \"expected_delivery\": \"2025-03-20\" } }"
            }
        ]
    },
    {
        "createdAt": "2025-03-14T06:15:05Z",
        "run_id": "0",
        "role": "assistant",
        "content": [
            {
                "type": "text",
                "text": "The order with ID 123 has been shipped and is expected to be delivered on March 15, 2025. However, the order with ID 124 is delayed and should now arrive by March 20, 2025. Is there anything else I can help you with?"
            }
        ]
    }
]

#please note that the tool definitions are not strictly required, and that some of the tools below are not used in the example above and that is ok.
#if context length is a concern you can remove the unused tool definitions or even the tool definitions altogether as the impact to the intent resolution evaluation is usual minimal.
tool_definitions = [
    {
        "name": "get_orders",
        "description": "Get the list of orders for a given account number.",
        "parameters": {
            "type": "object",
            "properties": {
                "account_number": {
                    "type": "string",
                    "description": "The account number to get the orders for."
                }
            }
        }
    },
    {
        "name": "get_order",
        "description": "Get the details of a specific order.",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "The order ID to get the details for."
                }
            }
        }
    },
    {
        "name": "initiate_return",
        "description": "Initiate the return process for an order.",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "The order ID for the return process."
                }
            }
        }
    },
    {
        "name": "update_shipping_address",
        "description": "Update the shipping address for a given account.",
        "parameters": {
            "type": "object",
            "properties": {
                "account_number": {
                    "type": "string",
                    "description": "The account number to update."
                },
                "new_address": {
                    "type": "string",
                    "description": "The new shipping address."
                }
            }
        }
    }
]

result = intent_resolution_evaluator(query            = query,
                                     response         = response,
                                     tool_definitions = tool_definitions,
                                    )
pprint(result)

{'gpt_intent_resolution': 5.0,
 'intent_resolution': 5.0,
 'intent_resolution_completion_tokens': 43,
 'intent_resolution_finish_reason': 'stop',
 'intent_resolution_model': 'gpt-4.1-nano-2025-04-14',
 'intent_resolution_prompt_tokens': 1999,
 'intent_resolution_reason': "The agent acknowledged the user's request and "
                             'provided detailed status updates for both '
                             'orders, fully resolving the intent to get order '
                             'statuses with accurate information.',
 'intent_resolution_result': 'pass',
 'intent_resolution_sample_input': '[{"role": "user", "content": '
                                   '"{\\"query\\": \\"User turn 1:\\\\n  Hi, I '
                                   'need help with the last 2 orders on my '
                                   'account #888. Could you please update me '
                                   'on their status?\\\\n\\\\n\\", '
                                   '\\"response

### Evaluating an agent conversation loaded from disk

In [5]:
import json
from azure.ai.evaluation import AIAgentConverter

def load_conversations(filename):
    with open(filename, 'r') as file:
        lines = file.readlines()
        parsed_conversations = [json.loads(line) for line in lines]
    print(f"Loaded {len(parsed_conversations)} conversations from {filename}.")
    return parsed_conversations

def get_run_ids_from_conversation(conversation):
    """Extract unique run_ids from a conversation"""
    run_ids = set()
    for message in conversation.get('messages', []):
        if 'run_id' in message:
            run_ids.add(message['run_id'])
    return sorted(list(run_ids))

def convert_conversation_for_evaluation(conversation, run_id):
    """
    Convert a conversation to the format expected by the evaluator.
    This is a workaround that manually constructs the output to avoid 
    the ToolDefinition validation error in _convert_from_conversation.
    """
    messages = conversation.get('messages', [])
    tools = conversation.get('tools', [])
    
    # Build query (all messages before and up to the current run_id)
    query = []
    response = []
    
    for message in messages:
        msg_run_id = message.get('run_id')
        role = message.get('role')
        content = message.get('content')
        
        # Add system messages to query
        if role == 'system':
            query.append({
                'role': 'system',
                'content': content
            })
        # Add user messages to query
        elif role == 'user':
            query.append({
                'role': 'user',
                'content': content if isinstance(content, str) else content,
                'createdAt': message.get('createdAt')
            })
        # Add assistant/tool messages based on run_id
        elif msg_run_id is not None:
            if str(msg_run_id) == str(run_id):
                # This is part of the current run's response
                response.append({
                    'role': role,
                    'content': content,
                    'run_id': msg_run_id,
                    'createdAt': message.get('createdAt')
                })
                # Also include tool_call_id if present
                if 'tool_call_id' in message:
                    response[-1]['tool_call_id'] = message['tool_call_id']
            elif int(msg_run_id) < int(run_id):
                # Previous run messages go into query
                query.append({
                    'role': role,
                    'content': content,
                    'run_id': msg_run_id,
                    'createdAt': message.get('createdAt')
                })
                if 'tool_call_id' in message:
                    query[-1]['tool_call_id'] = message['tool_call_id']
    
    # Build tool_definitions with proper structure including type
    tool_definitions = []
    for tool in tools:
        tool_def = {
            'type': tool.get('type', 'function'),
            'name': tool['name'],
            'description': tool.get('description', ''),
            'parameters': tool.get('parameters', {})
        }
        tool_definitions.append(tool_def)
    
    return {
        'query': query,
        'response': response,
        'tool_definitions': tool_definitions
    }

conversations_filename = r'sample_synthetic_conversations.jsonl'

#this loads 90 conversations from the file sample_synthetic_conversations.jsonl
sample_conversations = load_conversations(conversations_filename)

#get the first conversation from the loaded conversations
conversation = sample_conversations[10]

run_ids = get_run_ids_from_conversation(conversation)
print(f"Run IDs in conversation: {run_ids}")
run_id = str(run_ids[0]) # convert run_id to string in case it is some other type, e.g. an int
converted_conv = convert_conversation_for_evaluation(conversation, run_id)
# Extract the query and response from the conversation
query = converted_conv['query']
response = converted_conv['response']
tool_definitions = converted_conv['tool_definitions']

print(f"Run ID: {run_id}")
print(f"Query: {query}")
print(f"Response: {response}")
print(f"Tool Definitions: {tool_definitions}")

result = intent_resolution_evaluator(query = query, response = response, tool_definitions = tool_definitions)
print(f"Evaluation result")
pprint(result)

Loaded 90 conversations from sample_synthetic_conversations.jsonl.
Run IDs in conversation: [0, 1, 2]
Run ID: 0
Query: [{'role': 'system', 'content': 'You are a healthcare support agent assisting patients with appointment scheduling, prescription refills, test results, and general health inquiries.'}, {'role': 'user', 'content': [{'type': 'text', 'text': 'Can you update my health records? I recently had a lab test and need the results added to my profile.'}], 'createdAt': 1741618732}, {'role': 'user', 'content': [{'type': 'text', 'text': 'Yes, I have a follow-up question regarding my request.'}], 'createdAt': 1741618752}, {'role': 'user', 'content': [{'type': 'text', 'text': 'Thank you for your help.'}], 'createdAt': 1741618767}]
Response: [{'role': 'assistant', 'content': [{'type': 'text', 'text': 'I’ll update your health records with the new lab test results. One moment, please.'}], 'run_id': 0, 'createdAt': 1741618737}]
Tool Definitions: [{'type': 'function', 'name': 'schedule_appoi

# Putting it all together and evaluate an entire conversation run by run

In [6]:
def evaluate_conversation_run(conversation : dict, run_id : str, verbose=False):
    converted_conv = convert_conversation_for_evaluation(conversation, str(run_id))
    # Extract the query and response from the conversation
    query = converted_conv['query']
    response = converted_conv['response']
    tool_definitions = converted_conv['tool_definitions']
    
    if verbose:
        print(f"*********************************************")
        print(f"Evaluating conversation run with ID: {run_id}")
        print(f"Run ID: {run_id}")
        print(f"Query: {query}")
        print(f"Response: {response}")
        print(f"Tool Definitions: {tool_definitions}")

    # Evaluate the query and response using the intent resolution evaluator
    evaluation_result = intent_resolution_evaluator(query = query, response = response, tool_definitions = tool_definitions)
    if verbose:
        print(f"Evaluation Result:")
        pprint(evaluation_result)

    return evaluation_result

def evaluate_conversation(conversation, verbose=False):
    run_ids = get_run_ids_from_conversation(conversation)
    print(f"Runs available in conversation: {run_ids}")
    results = []
    for run_id in run_ids:
        result = evaluate_conversation_run(conversation, str(run_id), verbose)
        results.append(result)
    return results

sample_conversation = sample_conversations[20]
evaluate_conversation(sample_conversation, verbose=True)

Runs available in conversation: [0, 1, 2]
*********************************************
Evaluating conversation run with ID: 0
Run ID: 0
Query: [{'role': 'system', 'content': 'You are an insurance claims processing agent. You help customers file claims, check claim statuses, and update claim details.'}, {'role': 'user', 'content': [{'type': 'text', 'text': 'What is the current status of my claim number CLM12345?'}], 'createdAt': 1741618732}, {'role': 'user', 'content': [{'type': 'text', 'text': 'Yes, I have a follow-up question regarding my request.'}], 'createdAt': 1741618752}, {'role': 'user', 'content': [{'type': 'text', 'text': 'Thank you for your help.'}], 'createdAt': 1741618782}]
Response: [{'role': 'assistant', 'content': [{'type': 'text', 'text': 'I’m checking the status of your claim CLM12345 now. One moment please.'}], 'run_id': 0, 'createdAt': 1741618737}]
Tool Definitions: [{'type': 'function', 'name': 'file_claim', 'description': 'File a new insurance claim.', 'parameters

[{'intent_resolution': 3.0,
  'gpt_intent_resolution': 3.0,
  'intent_resolution_result': 'pass',
  'intent_resolution_threshold': 3,
  'intent_resolution_reason': 'The user asked to check the status of their claim. The agent acknowledged and initiated the check, but did not provide the actual status or confirm completion, resulting in an incomplete resolution of the intent.',
  'intent_resolution_prompt_tokens': 1936,
  'intent_resolution_completion_tokens': 55,
  'intent_resolution_total_tokens': 1991,
  'intent_resolution_finish_reason': 'stop',
  'intent_resolution_model': 'gpt-4.1-nano-2025-04-14',
  'intent_resolution_sample_input': '[{"role": "user", "content": "{\\"query\\": \\"User turn 1:\\\\n  What is the current status of my claim number CLM12345?\\\\n  Yes, I have a follow-up question regarding my request.\\\\n  Thank you for your help.\\\\n\\\\n\\", \\"response\\": \\"I\\\\u2019m checking the status of your claim CLM12345 now. One moment please.\\", \\"tool_definitions\\"